# **Testing with mean pooling, urduhack roberta, without dimensionality reduction and with articles divided into batches for capturing complete article’s semantic meaning (headline col embeddings)**

In [1]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION AND STORAGE IN CHROMADB
# Updated for 10,000 records with headline, category, and content columns
# Using MEAN POOLING and chroma_db_mean_headline database
# SEMANTIC SEARCH ON HEADLINE COLUMN
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in ChromaDB for efficient retrieval in recommendation systems.

    Designed for large datasets (10,000+ records) with semantic search on HEADLINE.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_mean_headline"):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path: Path to store ChromaDB locally
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.chroma_db_path.mkdir(exist_ok=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Initialize ChromaDB client (persistent storage)
        print(f"Initializing ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(
            path=str(self.chroma_db_path)
        )

        # Create or get collection for storing embeddings
        self.collection = self.client.get_or_create_collection(
            name="urdu_news_embeddings_10k_mean_headline",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity for recommendations
        )

    def mean_pooling(self, model_output, attention_mask):
        """
        Apply mean pooling to model output to get sentence embeddings.

        This takes the token embeddings and creates a single vector
        by taking the mean value across all tokens for each dimension,
        while ignoring padding tokens.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            Mean pooled embeddings (batch_size, embedding_dim)
        """
        # Extract token embeddings
        token_embeddings = model_output[0]

        # Expand attention mask for broadcasting
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        # Sum embeddings along sequence dimension, ignoring padding tokens
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)

        # Count non-padding tokens for each sequence
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        # Calculate mean embeddings
        mean_embeddings = sum_embeddings / sum_mask

        return mean_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512) -> np.ndarray:
        """
        Generate embedding for a single text using mean pooling.

        For headlines, we typically don't need chunking since they are short,
        but the method is kept for consistency.

        Args:
            text: Input Urdu text (headline column)
            max_length: Maximum tokens (default: 512)

        Returns:
            Embedding vector as numpy array (768,)
        """
        # Tokenize and generate embedding
        encoded_input = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

        with torch.no_grad():
            model_output = self.model(**encoded_input)

        embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
        embeddings = embeddings.cpu().detach().numpy()
        return embeddings[0]

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       headline_column: str = "Headline",
                                       content_column: str = "content",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles in dataset and store in ChromaDB.

        Semantic search is performed on the HEADLINE column, while content
        and category are stored as metadata for display.

        Args:
            df: Dataframe containing articles with headline, category, and content
            headline_column: Name of column containing article headline (for embeddings)
            content_column: Name of column containing article content (metadata)
            category_column: Name of column containing article category (metadata)
        """
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Headline column: '{headline_column}' (used for semantic search)")
        print(f"Content column: '{content_column}' (stored as metadata)")
        print(f"Category column: '{category_column}' (stored as metadata)")
        print(f"Pooling method: MEAN POOLING")
        print(f"Database: chroma_db_mean_headline")
        print(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data for ChromaDB
        ids = []
        embeddings = []
        metadatas = []
        documents = []

        for idx, row in df.iterrows():
            # Show progress every 500 articles (more frequent for 10k dataset)
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed
                eta = (total_articles - idx - 1) / articles_per_sec
                print(f"Processed {idx + 1}/{total_articles} articles "
                      f"({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)")

            # Get HEADLINE for semantic search (this is the key change)
            headline_text = str(row[headline_column])

            # Skip empty headlines
            if len(headline_text.strip()) == 0:
                print(f"Warning: Skipping article {idx} - empty headline")
                continue

            try:
                # Generate embedding from HEADLINE (this is the key change)
                embedding = self.generate_embedding_for_text(headline_text)

                # Prepare data for ChromaDB
                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings.append(embedding.tolist())

                # Store HEADLINE as document (for display/search results)
                documents.append(headline_text)

                # Store metadata (content, category, and article index)
                content_text = str(row.get(content_column, ""))
                metadata = {
                    "article_index": idx,
                    "headline": headline_text,
                    "category": str(row.get(category_column, "Unknown")),
                    "content": content_text,  # Store full content in metadata
                    "content_length": len(content_text),
                    "pooling_method": "mean_pooling",
                    "search_column": "headline"  # Indicate what we're searching on
                }
                metadatas.append(metadata)

            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        # Add all embeddings to ChromaDB in batches (ChromaDB has max batch size limit)
        print(f"\n{'='*70}")
        print(f"STORING {len(ids)} EMBEDDINGS IN CHROMADB...")
        print(f"{'='*70}")

        # ChromaDB has a max batch size limit (~5000), so we add in batches
        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            self.collection.add(
                ids=ids[batch_idx:batch_end],
                embeddings=embeddings[batch_idx:batch_end],
                documents=documents[batch_idx:batch_end],  # Headlines stored as documents
                metadatas=metadatas[batch_idx:batch_end]
            )

        print(f"✓ All batches stored successfully!")

        total_time = time.time() - start_time
        print(f"\n✓ Successfully stored {len(ids)} embeddings in ChromaDB")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Average time per article: {total_time/len(ids):.4f} seconds")
        print(f"Processing speed: {len(ids)/total_time:.2f} articles/second")

    def search_similar_articles(self, query_text: str, n_results: int = 5) -> dict:
        """
        Search for similar articles using query text.

        Searches based on HEADLINE embeddings and returns results with
        headline, category, and full content.

        Args:
            query_text: Query text to find similar articles (matched against headlines)
            n_results: Number of similar articles to return

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding from headline
        query_embedding = self.generate_embedding_for_text(query_text)

        # Search in ChromaDB
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about the ChromaDB collection.

        Returns:
            Dictionary with collection information
        """
        count = self.collection.count()
        return {
            "total_embeddings": count,
            "collection_name": self.collection.name,
            "pooling_method": "MEAN POOLING",
            "embedding_dimension": 768,
            "search_column": "HEADLINE",  # Updated to reflect the change
            "db_path": str(self.chroma_db_path)
        }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # Load your preprocessed dataset
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM")
    print("USING MEAN POOLING AND chroma_db_mean_headline DATABASE")
    print("SEMANTIC SEARCH ON HEADLINE COLUMN")
    print("="*70)
    print("\nLoading dataset...")

    df = pd.read_csv("final_cleaned_urdu_news.csv")

    print(f"\nDataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset info:")
    print(f"  - Total articles: {len(df)}")
    print(f"  - Unique categories: {df['Category'].nunique()}")
    print(f"  - Categories: {df['Category'].unique().tolist()}")

    # Show sample data
    print(f"\n{'='*70}")
    print("SAMPLE DATA PREVIEW")
    print(f"{'='*70}")
    sample = df.iloc[0]
    print(f"Headline: {sample['Headline']}")  # Full headline since it's short
    print(f"Category: {sample['Category']}")
    print(f"Content preview: {sample['content'][:200]}...")
    print(f"Content length: {len(sample['content'])} characters")
    print(f"Headline length: {len(sample['Headline'])} characters")

    # Initialize embedding generator
    print(f"\n{'='*70}")
    print("INITIALIZING EMBEDDING GENERATOR")
    print(f"{'='*70}")

    embedder = UrduNewsEmbeddingGenerator(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_mean_headline"  # Updated path
    )

    # Generate embeddings and store in ChromaDB
    embedder.generate_embeddings_for_dataset(
        df=df,
        headline_column="Headline",    # Semantic search on HEADLINE (capital H)
        content_column="content",      # Store as metadata
        category_column="Category"     # Store as metadata (capital C)
    )

    # Display collection statistics
    print("\n" + "="*70)
    print("CHROMADB COLLECTION STATISTICS")
    print("="*70)
    stats = embedder.get_collection_stats()
    for key, value in stats.items():
        print(f"{key}: {value}")

    # Test: Search for similar articles with custom query
    print("\n" + "="*70)
    print("TESTING SEMANTIC SEARCH ON HEADLINE")
    print("="*70)

    # Custom query - search for headlines about mobile companies
    query = "موبائل کمپنیاں"
    print(f"\nQuery: {query}\n")

    results = embedder.search_similar_articles(
        query_text=query,
        n_results=5
    )

    print("Top 5 most relevant articles found:")
    print("="*70)

    if results['ids'] and len(results['ids']) > 0:
        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            print(f"\n{'='*70}")
            print(f"RESULT #{i+1}")
            print(f"{'='*70}")
            print(f"Article ID: {doc_id}")
            print(f"Similarity Score: {1 - distance:.4f} (higher is better)")
            print(f"\nHEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"Pooling Method: {metadata.get('pooling_method', 'mean_pooling')}")
            print(f"Search Column: {metadata.get('search_column', 'headline')}")

            # Get full content from metadata (we stored it there)
            full_content = metadata.get('content', '')

            # Display article preview (first 300 characters)
            print(f"\n--- CONTENT PREVIEW (First 300 chars) ---")
            print(full_content[:300] + "..." if len(full_content) > 300 else full_content)

            print(f"\n{'='*70}")
    else:
        print("No results found!")

    print("\n" + "="*70)
    print("✓ EMBEDDINGS GENERATION AND STORAGE COMPLETED SUCCESSFULLY!")
    print("✓ Using MEAN POOLING for embeddings")
    print("✓ Database stored as: chroma_db_mean_headline")
    print("✓ Semantic search is now available on the HEADLINE column")
    print("✓ Content and categories are stored as metadata")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM
USING MEAN POOLING AND chroma_db_mean_headline DATABASE
SEMANTIC SEARCH ON HEADLINE COLUMN

Loading dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 4
  - Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters
Headline length: 59 characters

INITIALIZING EMBEDDING GENERATOR
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing ChromaDB at: chroma_db_mean_headline

GENERATING EMBEDDINGS FOR 111853 ARTIC

# **Recommender System Using Mean Pooling & Headline Col**

In [2]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM
# Uses pre-stored ChromaDB embeddings with MEAN POOLING to generate recommendations
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path

class UrduNewsRecommender:
    """
    Recommendation system for Urdu news articles using pre-stored MEAN POOLING embeddings.
    Connects to existing ChromaDB and generates recommendations based on queries.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_mean_headline",
                 collection_name: str = "urdu_news_embeddings_10k_mean_headline"):
        """
        Initialize the recommender with pre-stored MEAN POOLING embeddings.

        Args:
            model_name: HuggingFace model identifier (same as used for embedding generation)
            chroma_db_path: Path to existing ChromaDB with mean pooling embeddings
            collection_name: Name of the collection with mean pooling embeddings
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer (for query embeddings)
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Connect to existing ChromaDB with MEAN POOLING embeddings
        print(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(path=str(self.chroma_db_path))

        # Get existing collection with MEAN POOLING embeddings
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"✓ Connected to collection: {collection_name}")
            print(f"✓ Total articles in database: {self.collection.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name}'")
            print(f"Make sure embeddings are generated first!")
            raise e

    def mean_pooling(self, model_output, attention_mask):
        """
        Apply MEAN POOLING to get sentence embeddings.

        This takes the token embeddings and creates a single vector
        by taking the mean value across all tokens for each dimension,
        while ignoring padding tokens.
        """
        # Extract token embeddings
        token_embeddings = model_output[0]

        # Expand attention mask for broadcasting
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        # Sum embeddings along sequence dimension, ignoring padding tokens
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)

        # Count non-padding tokens for each sequence
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        # Calculate mean embeddings
        mean_embeddings = sum_embeddings / sum_mask

        return mean_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for query text using MEAN POOLING.

        Args:
            query_text: Urdu query text
            max_length: Maximum tokens per chunk
            chunk_overlap: Overlap between chunks

        Returns:
            Query embedding vector using MEAN POOLING
        """
        # Tokenize to check length
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # If query fits in max_length, process normally
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long queries: use chunking (same as article processing)
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def get_recommendations(self, query: str, n_results: int = 5,
                          filter_category: str = None) -> dict:
        """
        Get article recommendations based on Urdu query using MEAN POOLING embeddings.

        Args:
            query: Urdu text query
            n_results: Number of recommendations to return
            filter_category: Optional category filter (e.g., "Sports", "Business")

        Returns:
            Dictionary containing recommended articles with metadata
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS USING MEAN POOLING")
        print(f"{'='*70}")
        print(f"Query: {query}")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Generate query embedding using MEAN POOLING
        print("Generating query embedding using MEAN POOLING...")
        query_embedding = self.generate_query_embedding(query)
        print("✓ Query embedding generated")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Search in ChromaDB with MEAN POOLING embeddings
        print(f"Searching for similar articles...")
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"✓ Found {len(results['ids'][0]) if results['ids'] else 0} recommendations\n")

        return results

    def display_recommendations(self, results: dict, df: pd.DataFrame = None,
                               show_full_content: bool = False):
        """
        Display recommendations in a formatted way.

        Args:
            results: Results from get_recommendations()
            df: Optional dataframe to fetch full content
            show_full_content: Whether to display full article content
        """
        if not results['ids'] or len(results['ids'][0]) == 0:
            print("❌ No recommendations found!")
            return

        print(f"{'='*70}")
        print(f"TOP {len(results['ids'][0])} RECOMMENDATIONS")
        print(f"{'='*70}\n")

        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            # Calculate similarity score (1 - distance for cosine)
            similarity_score = 1 - distance

            print(f"{'='*70}")
            print(f"RECOMMENDATION #{i+1}")
            print(f"{'='*70}")
            print(f"📰 Article ID: {doc_id}")
            print(f"🎯 Similarity Score: {similarity_score:.4f} ({similarity_score*100:.2f}%)")
            print(f"\n📌 HEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"📂 CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"📏 Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"🔧 Pooling Method: {metadata.get('pooling_method', 'mean_pooling')}")

            # Get and display article content
            article_idx = metadata.get('article_index', None)

            if df is not None and article_idx is not None and article_idx < len(df):
                full_content = df.iloc[article_idx]['content']

                if show_full_content:
                    print(f"\n📄 FULL CONTENT:")
                    print(f"{'-'*70}")
                    print(full_content)
                else:
                    # Display preview (first 400 characters)
                    preview = full_content[:400] + "..." if len(full_content) > 400 else full_content
                    print(f"\n📄 CONTENT PREVIEW:")
                    print(f"{'-'*70}")
                    print(preview)
            else:
                # Fallback to stored document preview
                print(f"\n📄 CONTENT PREVIEW:")
                print(f"{'-'*70}")
                print(document)

            print(f"\n{'='*70}\n")

    def get_statistics(self) -> dict:
        """Get statistics about the recommendation system."""
        return {
            "total_articles": self.collection.count(),
            "collection_name": self.collection.name,
            "model": self.model_name,
            "device": str(self.device),
            "embedding_dimension": 768,
            "pooling_method": "MEAN POOLING"
        }


# =============================================================================
# MAIN EXECUTION - RECOMMENDATION SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM - MEAN POOLING EMBEDDINGS")
    print("="*70)

    # Load the dataset (optional - only needed for displaying full content)
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender (connects to existing ChromaDB with MEAN POOLING embeddings)
    print("\n" + "="*70)
    print("INITIALIZING RECOMMENDATION SYSTEM WITH MEAN POOLING")
    print("="*70)

    recommender = UrduNewsRecommender(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_mean_headline",
        collection_name="urdu_news_embeddings_10k_mean_headline"
    )

    # Display system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    for key, value in stats.items():
        print(f"{key}: {value}")

    # =================================================================
    # EXAMPLE 1: Simple query-based recommendations
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 1: TECHNOLOGY NEWS RECOMMENDATION")
    print("="*70)

    query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"

    results = recommender.get_recommendations(
        query=query,
        n_results=10
    )

    recommender.display_recommendations(
        results=results,
        df=df,
        show_full_content=False  # Set to True to see full articles
    )

    # =================================================================
    # EXAMPLE 2: Category-filtered recommendations
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 2: SPORTS NEWS RECOMMENDATION (CATEGORY FILTERED)")
    print("="*70)

    query = " ان لائن ٹریفک نگرانی نئے حکومتی منصوبے خدشات کراچی حکومت جانب ان لائن ٹریفک ٹیلی کام انڈسٹری ڈیجیٹل حقوق وکالت کرنے والوں نگرانی منصوبے اس عمل شفافیت پرائیویسی حوالے کئی خدشات جنم لے ہیںڈان اخبار رپورٹ پاکستان ٹیلی کمیونیکیشن اتھارٹی پی ٹی اے حال گرے ٹریفک نگرانی تجزیے ٹیلی کام انڈسٹری مناسب تکینکی حل تلاش کرنے احکامات دیے تھےاس نظام نام ویب مانیٹرنگ سسٹم ہوگا جس تحت قومی سلامتی مقاصد روابط نگرانی ٹریفک ریکارڈ کرنے کال ڈیٹا ریکارڈ کرنے پی ٹی اے جانب فراہم گئی سروس میعار جانچنے اقدامات کیے جائیں گےیہ پڑھیں ان لائن غیر قانونی سرگرمیوں کارروائی کیلئے پی ٹی اے خصوصی شعبہ قائم واضح پی ٹی اے مانیٹرنگ ری کنسیلی ایشن اف انٹرنیشنل ٹیلی فون ٹریفک ریگولیشنز ایم ار ائی ٹی ٹی تحت ان لائن ٹریفک نگرانی اسے بلاک کرنے اختیار حاصل بھلے انکریپٹڈ ہوں اس پاکستان شروع ہونے ختم ہونے اوازیں ڈیٹا شامل اس تمام انکرپٹڈ وی او ائی پی وائس اوور ائی پی سروسز شامل ہیںسینیٹ قائمہ کمیٹی برائے کابینہ سیکریٹریٹ انٹرنیٹ ٹریفک نگرانی پی ٹی اے امریکی کمپنی سینڈوین کوواپریشن درمیان ہونے سمجھوتے خدشات اظہار کیا مبینہ طور اسرائیلی خفیہ تنظیم شراکت دار طور کام کرتی ہےاس سمجھوتے تحت امریکی کمپنی واٹس ایپ سمیت پاکستانیوں تمام تر ڈیجیٹل معلومات رسائی حاصل ہوجائے گیاس حوالے گفتگو کرتے وزیراعظم ٹاسک فورس برائے ائی اینڈ ٹیلی کام رکن وہاج السراج کہنا اس نظام سختی عمل کیا گیا واٹس ایپ فیس ٹائم میسجنر جیسی سروسز بلاک کرنے قانونی بنایا جاسکتا مزید پڑھیں سوشل میڈیا اپ نگرانی ہو اس بچا کیسے جائے انہوں بتایا پی ٹی اے بڑے پیمانے انٹرنیٹ مانیٹرنگ ضرورت اس تحت ادارے وی پی این سروسز بلاک کرنے اجازت ہوگی انٹرنیٹ ٹریفک نگرانی مداخلت تصور جاتیں ہیںیاد وضع کردہ پاکستان الیکٹرونک کرائمز ایکٹ پیکا قبل ویب سائٹ بلاک کرنے درخواست وزارت کمیٹی ذریعے جاتی پی ٹی اے ہدایت کرتی انٹرنیٹ سروس فراہم کرنے والوں ائی ایس پیز متعلقہ ویب سائٹ بلاک کرنے حکم جائےتاہم پیکا تحت پی ٹی اے تمام قابل اعتراض مواد جس قانون تحت تفصیلی تشریح بیان گئی براہ راست بلاک کرنے اختیارات حاصل ہیںحکام پاکستان انٹرنیٹ ایکسچینج پی ائی ای ذریعے یو ار ایل بلاک سینسر کرسکتے جس بعد ائی ایس پیز پی ٹی اے جانب جاری کردہ ان ہدایات پیروی کرنی ہوگی بصورت دیگر ان لائسنس منسوخ کردیا جائے گا "
    results = recommender.get_recommendations(
        query=query,
        n_results=10,

    )

    recommender.display_recommendations(
        results=results,
        df=df,
        show_full_content=False
    )

    # =================================================================
    # EXAMPLE 3: Multiple queries (batch recommendations)
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 3: MULTIPLE QUERY RECOMMENDATIONS")
    print("="*70)

    queries = [
        "معیشت اور کاروبار کی خبریں",
        "کرکٹ کی تازہ ترین خبریں",
        "فلموں اور ڈرامے کی خبریں",
        "پاکستان کا انتخابی نظام",
        "پاکستانی شوبز انڈسٹری",
        "صحت اور تندرستی کے حوالے سے مفید معلومات",
        "پاکستان میں تعلیمی نظام اور جدید تربیت",
        "ٹیکنالوجی",
        "کاروبار اور معاشی ترقی کی خبریں",
        "پاکستانی سیاست اور حکومتی پالیسیاں",
        "کھیلوں اور تفریحی پروگراموں کی خبریں",
        "مذہبی تعلیمات اور روحانی معلومات",
        "پاکستان کے خوبصورت سیاحتی مقامات"

    ]



    for idx, query in enumerate(queries, 1):
        print(f"\n{'─'*70}")
        print(f"QUERY {idx}: {query}")
        print(f"{'─'*70}")

        results = recommender.get_recommendations(
            query=query,
            n_results=10  # Get top 10 for each query
        )

        # Display only headlines for compact view
        if results['ids'] and len(results['ids'][0]) > 0:
            for i, (doc_id, distance, metadata) in enumerate(zip(
                results['ids'][0],
                results['distances'][0],
                results['metadatas'][0]
            ), 1):
                similarity = (1 - distance) * 100
                print(f"{i}. [{similarity:.1f}%] {metadata.get('headline', 'N/A')}")
        print()

    print("="*70)
    print("✓ RECOMMENDATION SYSTEM DEMO COMPLETED!")
    print("✓ Using MEAN POOLING embeddings for recommendations")
    print("✓ Connected to chroma_db_mean database")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM - MEAN POOLING EMBEDDINGS

Loading dataset for content display...
✓ Dataset loaded: 111853 articles

INITIALIZING RECOMMENDATION SYSTEM WITH MEAN POOLING
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Connecting to ChromaDB at: chroma_db_mean_headline
✓ Connected to collection: urdu_news_embeddings_10k_mean_headline
✓ Total articles in database: 111853

SYSTEM STATISTICS
total_articles: 111853
collection_name: urdu_news_embeddings_10k_mean_headline
model: urduhack/roberta-urdu-small
device: cuda
embedding_dimension: 768
pooling_method: MEAN POOLING


EXAMPLE 1: TECHNOLOGY NEWS RECOMMENDATION

GENERATING RECOMMENDATIONS USING MEAN POOLING
Query: پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن
Number of results: 10

Generating query embedding using MEAN POOLING...
✓ Query embedding generated
Searching for similar articles...
✓ Found 10 recommendations

TOP 10 RECOMMENDATIONS

RECOMMENDATION #1
📰 Article ID: article_7
🎯 Similari

# **Testing MAX pooling, urduhack roberta, without dimensionality reduction and with articles divided into batches for capturing complete article’s semantic meaning (headline col embeddings)**

In [3]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION AND STORAGE IN CHROMADB
# Updated for 10,000 records with headline, category, and content columns
# Using MAX POOLING and chroma_db_max_headline database
# SEMANTIC SEARCH ON HEADLINE COLUMN
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in ChromaDB for efficient retrieval in recommendation systems.

    Designed for large datasets (10,000+ records) with semantic search on HEADLINE.
    Uses MAX POOLING for embedding generation.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_max_headline"):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path: Path to store ChromaDB locally
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.chroma_db_path.mkdir(exist_ok=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Initialize ChromaDB client (persistent storage)
        print(f"Initializing ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(
            path=str(self.chroma_db_path)
        )

        # Create or get collection for storing embeddings
        self.collection = self.client.get_or_create_collection(
            name="urdu_news_embeddings_10k_max_headline",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity for recommendations
        )

    def max_pooling(self, model_output, attention_mask):
        """
        Apply max pooling to model output to get sentence embeddings.

        This takes the token embeddings and creates a single vector
        by taking the maximum value across all tokens for each dimension,
        while ignoring padding tokens.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            Max pooled embeddings (batch_size, embedding_dim)
        """
        # Extract token embeddings
        token_embeddings = model_output[0]

        # Expand attention mask for broadcasting
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        # Set padding tokens to very small value so they don't affect max pooling
        token_embeddings[input_mask_expanded == 0] = -1e9

        # Apply max pooling along sequence dimension
        max_embeddings = torch.max(token_embeddings, 1)[0]

        return max_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512) -> np.ndarray:
        """
        Generate embedding for a single text using MAX pooling.

        For headlines, we typically don't need chunking since they are short,
        but the method is kept for consistency.

        Args:
            text: Input Urdu text (headline column)
            max_length: Maximum tokens (default: 512)

        Returns:
            Embedding vector as numpy array (768,)
        """
        # Tokenize and generate embedding
        encoded_input = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

        with torch.no_grad():
            model_output = self.model(**encoded_input)

        # Use MAX pooling instead of mean pooling
        embeddings = self.max_pooling(model_output, encoded_input['attention_mask'])
        embeddings = embeddings.cpu().detach().numpy()
        return embeddings[0]

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       headline_column: str = "Headline",
                                       content_column: str = "content",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles in dataset and store in ChromaDB.

        Semantic search is performed on the HEADLINE column, while content
        and category are stored as metadata for display.

        Args:
            df: Dataframe containing articles with headline, category, and content
            headline_column: Name of column containing article headline (for embeddings)
            content_column: Name of column containing article content (metadata)
            category_column: Name of column containing article category (metadata)
        """
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Headline column: '{headline_column}' (used for semantic search)")
        print(f"Content column: '{content_column}' (stored as metadata)")
        print(f"Category column: '{category_column}' (stored as metadata)")
        print(f"Pooling method: MAX POOLING")
        print(f"Database: chroma_db_max_headline")
        print(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data for ChromaDB
        ids = []
        embeddings = []
        metadatas = []
        documents = []

        for idx, row in df.iterrows():
            # Show progress every 500 articles (more frequent for 10k dataset)
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed
                eta = (total_articles - idx - 1) / articles_per_sec
                print(f"Processed {idx + 1}/{total_articles} articles "
                      f"({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)")

            # Get HEADLINE for semantic search
            headline_text = str(row[headline_column])

            # Skip empty headlines
            if len(headline_text.strip()) == 0:
                print(f"Warning: Skipping article {idx} - empty headline")
                continue

            try:
                # Generate embedding from HEADLINE using MAX pooling
                embedding = self.generate_embedding_for_text(headline_text)

                # Prepare data for ChromaDB
                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings.append(embedding.tolist())

                # Store HEADLINE as document (for display/search results)
                documents.append(headline_text)

                # Store metadata (content, category, and article index)
                content_text = str(row.get(content_column, ""))
                metadata = {
                    "article_index": idx,
                    "headline": headline_text,
                    "category": str(row.get(category_column, "Unknown")),
                    "content": content_text,  # Store full content in metadata
                    "content_length": len(content_text),
                    "pooling_method": "max_pooling",  # Updated to max pooling
                    "search_column": "headline"  # Indicate what we're searching on
                }
                metadatas.append(metadata)

            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        # Add all embeddings to ChromaDB in batches (ChromaDB has max batch size limit)
        print(f"\n{'='*70}")
        print(f"STORING {len(ids)} EMBEDDINGS IN CHROMADB...")
        print(f"{'='*70}")

        # ChromaDB has a max batch size limit (~5000), so we add in batches
        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            self.collection.add(
                ids=ids[batch_idx:batch_end],
                embeddings=embeddings[batch_idx:batch_end],
                documents=documents[batch_idx:batch_end],  # Headlines stored as documents
                metadatas=metadatas[batch_idx:batch_end]
            )

        print(f"✓ All batches stored successfully!")

        total_time = time.time() - start_time
        print(f"\n✓ Successfully stored {len(ids)} embeddings in ChromaDB")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Average time per article: {total_time/len(ids):.4f} seconds")
        print(f"Processing speed: {len(ids)/total_time:.2f} articles/second")

    def search_similar_articles(self, query_text: str, n_results: int = 5) -> dict:
        """
        Search for similar articles using query text.

        Searches based on HEADLINE embeddings and returns results with
        headline, category, and full content.

        Args:
            query_text: Query text to find similar articles (matched against headlines)
            n_results: Number of similar articles to return

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding from headline using MAX pooling
        query_embedding = self.generate_embedding_for_text(query_text)

        # Search in ChromaDB
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about the ChromaDB collection.

        Returns:
            Dictionary with collection information
        """
        count = self.collection.count()
        return {
            "total_embeddings": count,
            "collection_name": self.collection.name,
            "pooling_method": "MAX POOLING",  # Updated to max pooling
            "embedding_dimension": 768,
            "search_column": "HEADLINE",
            "db_path": str(self.chroma_db_path)
        }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # Load your preprocessed dataset
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM")
    print("USING MAX POOLING AND chroma_db_max_headline DATABASE")
    print("SEMANTIC SEARCH ON HEADLINE COLUMN")
    print("="*70)
    print("\nLoading balanced dataset...")

    df = pd.read_csv("final_cleaned_urdu_news.csv")

    print(f"\nDataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset info:")
    print(f"  - Total articles: {len(df)}")
    print(f"  - Unique categories: {df['Category'].nunique()}")
    print(f"  - Categories: {df['Category'].unique().tolist()}")

    # Show sample data
    print(f"\n{'='*70}")
    print("SAMPLE DATA PREVIEW")
    print(f"{'='*70}")
    sample = df.iloc[0]
    print(f"Headline: {sample['Headline']}")  # Full headline since it's short
    print(f"Category: {sample['Category']}")
    print(f"Content preview: {sample['content'][:200]}...")
    print(f"Content length: {len(sample['content'])} characters")
    print(f"Headline length: {len(sample['Headline'])} characters")

    # Initialize embedding generator
    print(f"\n{'='*70}")
    print("INITIALIZING EMBEDDING GENERATOR")
    print(f"{'='*70}")

    embedder = UrduNewsEmbeddingGenerator(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_max_headline"  # Updated path to max_headline
    )

    # Generate embeddings and store in ChromaDB
    embedder.generate_embeddings_for_dataset(
        df=df,
        headline_column="Headline",    # Semantic search on HEADLINE (capital H)
        content_column="content",      # Store as metadata
        category_column="Category"     # Store as metadata (capital C)
    )

    # Display collection statistics
    print("\n" + "="*70)
    print("CHROMADB COLLECTION STATISTICS")
    print("="*70)
    stats = embedder.get_collection_stats()
    for key, value in stats.items():
        print(f"{key}: {value}")

    # Test: Search for similar articles with custom query
    print("\n" + "="*70)
    print("TESTING SEMANTIC SEARCH ON HEADLINE")
    print("="*70)

    # Custom query - search for headlines about mobile companies
    query = "موبائل کمپنیاں"
    print(f"\nQuery: {query}\n")

    results = embedder.search_similar_articles(
        query_text=query,
        n_results=5
    )

    print("Top 5 most relevant articles found:")
    print("="*70)

    if results['ids'] and len(results['ids']) > 0:
        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            print(f"\n{'='*70}")
            print(f"RESULT #{i+1}")
            print(f"{'='*70}")
            print(f"Article ID: {doc_id}")
            print(f"Similarity Score: {1 - distance:.4f} (higher is better)")
            print(f"\nHEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"Pooling Method: {metadata.get('pooling_method', 'max_pooling')}")  # Updated
            print(f"Search Column: {metadata.get('search_column', 'headline')}")

            # Get full content from metadata (we stored it there)
            full_content = metadata.get('content', '')

            # Display article preview (first 300 characters)
            print(f"\n--- CONTENT PREVIEW (First 300 chars) ---")
            print(full_content[:300] + "..." if len(full_content) > 300 else full_content)

            print(f"\n{'='*70}")
    else:
        print("No results found!")

    print("\n" + "="*70)
    print("✓ EMBEDDINGS GENERATION AND STORAGE COMPLETED SUCCESSFULLY!")
    print("✓ Using MAX POOLING for embeddings")
    print("✓ Database stored as: chroma_db_max_headline")
    print("✓ Semantic search is now available on the HEADLINE column")
    print("✓ Content and categories are stored as metadata")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM
USING MAX POOLING AND chroma_db_max_headline DATABASE
SEMANTIC SEARCH ON HEADLINE COLUMN

Loading balanced dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 4
  - Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters
Headline length: 59 characters

INITIALIZING EMBEDDING GENERATOR
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing ChromaDB at: chroma_db_max_headline

GENERATING EMBEDDINGS FOR 111853

# **Recommender System for MAX POOLING**


In [2]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM
# Uses pre-stored ChromaDB embeddings with MAX POOLING to generate recommendations
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path

class UrduNewsRecommender:
    """
    Recommendation system for Urdu news articles using pre-stored MAX POOLING embeddings.
    Connects to existing ChromaDB and generates recommendations based on queries.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_max_headline",
                 collection_name: str = "uurdu_news_embeddings_10k_max_headline"):
        """
        Initialize the recommender with pre-stored MAX POOLING embeddings.

        Args:
            model_name: HuggingFace model identifier (same as used for embedding generation)
            chroma_db_path: Path to existing ChromaDB with max pooling embeddings
            collection_name: Name of the collection with max pooling embeddings
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer (for query embeddings)
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Connect to existing ChromaDB with MAX POOLING embeddings
        print(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(path=str(self.chroma_db_path))

        # Get existing collection with MAX POOLING embeddings
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"✓ Connected to collection: {collection_name}")
            print(f"✓ Total articles in database: {self.collection.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name}'")
            print(f"Make sure embeddings are generated first!")
            raise e

    def max_pooling(self, model_output, attention_mask):
        """Apply MAX POOLING to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        token_embeddings[input_mask_expanded == 0] = -1e9
        max_embeddings = torch.max(token_embeddings, 1)[0]
        return max_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for query text using MAX POOLING.

        Args:
            query_text: Urdu query text
            max_length: Maximum tokens per chunk
            chunk_overlap: Overlap between chunks

        Returns:
            Query embedding vector using MAX POOLING
        """
        # Tokenize to check length
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # If query fits in max_length, process normally
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.max_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long queries: use chunking (same as article processing)
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.max_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def get_recommendations(self, query: str, n_results: int = 5,
                          filter_category: str = None) -> dict:
        """
        Get article recommendations based on Urdu query using MAX POOLING embeddings.

        Args:
            query: Urdu text query
            n_results: Number of recommendations to return
            filter_category: Optional category filter (e.g., "Sports", "Business")

        Returns:
            Dictionary containing recommended articles with metadata
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS USING MAX POOLING")
        print(f"{'='*70}")
        print(f"Query: {query}")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Generate query embedding using MAX POOLING
        print("Generating query embedding using MAX POOLING...")
        query_embedding = self.generate_query_embedding(query)
        print("✓ Query embedding generated")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Search in ChromaDB with MAX POOLING embeddings
        print(f"Searching for similar articles...")
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"✓ Found {len(results['ids'][0]) if results['ids'] else 0} recommendations\n")

        return results

    def display_recommendations(self, results: dict, df: pd.DataFrame = None,
                               show_full_content: bool = False):
        """
        Display recommendations in a formatted way.

        Args:
            results: Results from get_recommendations()
            df: Optional dataframe to fetch full content
            show_full_content: Whether to display full article content
        """
        if not results['ids'] or len(results['ids'][0]) == 0:
            print("❌ No recommendations found!")
            return

        print(f"{'='*70}")
        print(f"TOP {len(results['ids'][0])} RECOMMENDATIONS")
        print(f"{'='*70}\n")

        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            # Calculate similarity score (1 - distance for cosine)
            similarity_score = 1 - distance

            print(f"{'='*70}")
            print(f"RECOMMENDATION #{i+1}")
            print(f"{'='*70}")
            print(f"📰 Article ID: {doc_id}")
            print(f"🎯 Similarity Score: {similarity_score:.4f} ({similarity_score*100:.2f}%)")
            print(f"\n📌 HEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"📂 CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"📏 Content Length: {metadata.get('content_length', 'N/A')} characters")

            # Get and display article content
            article_idx = metadata.get('article_index', None)

            if df is not None and article_idx is not None and article_idx < len(df):
                full_content = df.iloc[article_idx]['content']

                if show_full_content:
                    print(f"\n📄 FULL CONTENT:")
                    print(f"{'-'*70}")
                    print(full_content)
                else:
                    # Display preview (first 400 characters)
                    preview = full_content[:400] + "..." if len(full_content) > 400 else full_content
                    print(f"\n📄 CONTENT PREVIEW:")
                    print(f"{'-'*70}")
                    print(preview)
            else:
                # Fallback to stored document preview
                print(f"\n📄 CONTENT PREVIEW:")
                print(f"{'-'*70}")
                print(document)

            print(f"\n{'='*70}\n")

    def get_statistics(self) -> dict:
        """Get statistics about the recommendation system."""
        return {
            "total_articles": self.collection.count(),
            "collection_name": self.collection.name,
            "model": self.model_name,
            "device": str(self.device),
            "embedding_dimension": 768
        }


# =============================================================================
# MAIN EXECUTION - RECOMMENDATION SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM - MAX POOLING EMBEDDINGS")
    print("="*70)

    # Load the dataset (optional - only needed for displaying full content)
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender (connects to existing ChromaDB with MAX POOLING embeddings)
    print("\n" + "="*70)
    print("INITIALIZING RECOMMENDATION SYSTEM WITH MAX POOLING")
    print("="*70)

    recommender = UrduNewsRecommender(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_max_headline",
        collection_name="urdu_news_embeddings_10k_max_headline"
    )

    # Display system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    for key, value in stats.items():
        print(f"{key}: {value}")

    # =================================================================
    # EXAMPLE 1: Simple query-based recommendations
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 1: TECHNOLOGY NEWS RECOMMENDATION")
    print("="*70)

    query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"

    results = recommender.get_recommendations(
        query=query,
        n_results=5
    )

    recommender.display_recommendations(
        results=results,
        df=df,
        show_full_content=False  # Set to True to see full articles
    )

    # =================================================================
    # EXAMPLE 2: Category-filtered recommendations
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 2: SPORTS NEWS RECOMMENDATION (CATEGORY FILTERED)")
    print("="*70)

    query = " پاکستان انٹرنیشنل کرکٹ ائی سی سی ائندہ ماہ اہم فیصلے امکان انٹرنیشنل کرکٹ کونسل پاکستان انٹرنیشنل کرکٹ واپسی تعلق ائندہ ماہ اجلاس اہم فیصلے کرسکتی پاکستان مجوزہ ائی سی سی ورلڈ الیون امد فیصلہ ائندہ ماہ ہونے امکان انٹرنیشنل کرکٹ کونسل ترجمان کہنا اپریل ہونے بورڈ اجلاس ائی سی سی ان مختلف بورڈ ممبران سیکیورٹی مینجرز رپورٹ جائزہ لے پی ایس ایل فائنل موقع لاہور موجود ترجمان بتایا رپورٹ روشنی بورڈ ممبران پاکستان کرکٹ واپسی حوالے ٹاسک فورس سفارشات مزید بات ہو پاکستان ٹاسک فورس چیرمین جائلز کلارک اس سال ستمبر چار ٹی ٹوینٹیز کیلئے ورلڈ الیون ٹیم پاکستان بھیجنے بات ترجمان ائی سی سی اس فیصلہ اپریل اجلاس ہوسکتا لاہور پی ایس ایل فائنل موقع ائی سی سی انگلینڈ اسٹریلیا سری لنکا بنگلہ دیش کرکٹ بورڈ سیکیورٹی مینجرز پاکستان موجود انٹرنیشنل کرکٹ کونسل پاکستان انٹرنیشنل کرکٹ واپسی تعلق ائندہ ماہ اجلاس اہم فیصلے کرسکتی پاکستان مجوزہ ائی سی سی ورلڈ الیون امد فیصلہ ائندہ ماہ ہونے امکان انٹرنیشنل کرکٹ کونسل ترجمان کہنا اپریل ہونے بورڈ اجلاس ائی سی سی"

    results = recommender.get_recommendations(
        query=query,
        n_results=5,

    )

    recommender.display_recommendations(
        results=results,
        df=df,
        show_full_content=False
    )

    # =================================================================
    # EXAMPLE 3: Multiple queries (batch recommendations)
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 3: MULTIPLE QUERY RECOMMENDATIONS")
    print("="*70)
 
    queries = [
        "معیشت اور کاروبار کی خبریں",
        "کرکٹ کی تازہ ترین خبریں",
        "فلموں اور ڈرامے کی خبریں",
        "پاکستان کا انتخابی نظام",
        "پاکستانی شوبز انڈسٹری",
        "صحت اور تندرستی کے حوالے سے مفید معلومات",
        "پاکستان میں تعلیمی نظام اور جدید تربیت",
        "ٹیکنالوجی",
        "کاروبار اور معاشی ترقی کی خبریں",
        "پاکستانی سیاست اور حکومتی پالیسیاں",
        "کھیلوں اور تفریحی پروگراموں کی خبریں",
        "مذہبی تعلیمات اور روحانی معلومات",
        "پاکستان کے خوبصورت سیاحتی مقامات"

    ]


    for idx, query in enumerate(queries, 1):
        print(f"\n{'─'*70}")
        print(f"QUERY {idx}: {query}")
        print(f"{'─'*70}")

        results = recommender.get_recommendations(
            query=query,
            n_results=10  # Get top 3 for each query
        )

        # Display only headlines for compact view
        if results['ids'] and len(results['ids'][0]) > 0:
            for i, (doc_id, distance, metadata) in enumerate(zip(
                results['ids'][0],
                results['distances'][0],
                results['metadatas'][0]
            ), 1):
                similarity = (1 - distance) * 100
                print(f"{i}. [{similarity:.1f}%] {metadata.get('headline', 'N/A')}")
        print()

    print("="*70)
    print("✓ RECOMMENDATION SYSTEM DEMO COMPLETED!")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM - MAX POOLING EMBEDDINGS

Loading dataset for content display...
✓ Dataset loaded: 111853 articles

INITIALIZING RECOMMENDATION SYSTEM WITH MAX POOLING
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Connecting to ChromaDB at: chroma_db_max_headline
✓ Connected to collection: urdu_news_embeddings_10k_max_headline
✓ Total articles in database: 111853

SYSTEM STATISTICS
total_articles: 111853
collection_name: urdu_news_embeddings_10k_max_headline
model: urduhack/roberta-urdu-small
device: cuda
embedding_dimension: 768


EXAMPLE 1: TECHNOLOGY NEWS RECOMMENDATION

GENERATING RECOMMENDATIONS USING MAX POOLING
Query: پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن
Number of results: 5

Generating query embedding using MAX POOLING...
✓ Query embedding generated
Searching for similar articles...
✓ Found 5 recommendations

TOP 5 RECOMMENDATIONS

RECOMMENDATION #1
📰 Article ID: article_7
🎯 Similarity Score: 1.0000 (100.00%)

📌 HEADLINE:

# **Testing with 10,000 dataset, CLS pooling, urduhack roberta, without dimensionality reduction and with articles divided into batches for capturing complete article’s semantic meaning (headline col embeddings)**

In [5]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION AND STORAGE IN CHROMADB
# Updated for 10,000 records with headline, category, and content columns
# Using CLS POOLING and chroma_db_cls_headline database
# SEMANTIC SEARCH ON HEADLINE COLUMN
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in ChromaDB for efficient retrieval in recommendation systems.

    Designed for large datasets (10,000+ records) with semantic search on HEADLINE.
    Uses CLS POOLING for embedding generation.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_cls_headline"):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path: Path to store ChromaDB locally
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.chroma_db_path.mkdir(exist_ok=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Initialize ChromaDB client (persistent storage)
        print(f"Initializing ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(
            path=str(self.chroma_db_path)
        )

        # Create or get collection for storing embeddings
        self.collection = self.client.get_or_create_collection(
            name="urdu_news_embeddings_10k_cls_headline",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity for recommendations
        )

    def cls_pooling(self, model_output, attention_mask=None):
        """
        Apply CLS pooling to model output to get sentence embeddings.

        This method extracts the [CLS] token embedding which is specifically
        trained to represent the entire sequence in transformer models.

        Args:
            model_output: Output from transformer model
            attention_mask: Not used in CLS pooling, kept for consistency

        Returns:
            CLS token embeddings (batch_size, embedding_dim)
        """
        # Extract token embeddings
        token_embeddings = model_output[0]

        # Extract the [CLS] token embedding (first token in the sequence)
        cls_embeddings = token_embeddings[:, 0, :]  # Shape: (batch_size, hidden_size)

        return cls_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512) -> np.ndarray:
        """
        Generate embedding for a single text using CLS pooling.

        For headlines, we typically don't need chunking since they are short,
        but the method is kept for consistency.

        Args:
            text: Input Urdu text (headline column)
            max_length: Maximum tokens (default: 512)

        Returns:
            Embedding vector as numpy array (768,)
        """
        # Tokenize and generate embedding
        encoded_input = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
            return_attention_mask=True,
            add_special_tokens=True  # Ensure [CLS] and [SEP] tokens are added
        )
        encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

        with torch.no_grad():
            model_output = self.model(**encoded_input)

        # Use CLS pooling instead of mean/max pooling
        embeddings = self.cls_pooling(model_output, encoded_input['attention_mask'])
        embeddings = embeddings.cpu().detach().numpy()
        return embeddings[0]

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       headline_column: str = "Headline",
                                       content_column: str = "content",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles in dataset and store in ChromaDB.

        Semantic search is performed on the HEADLINE column, while content
        and category are stored as metadata for display.

        Args:
            df: Dataframe containing articles with headline, category, and content
            headline_column: Name of column containing article headline (for embeddings)
            content_column: Name of column containing article content (metadata)
            category_column: Name of column containing article category (metadata)
        """
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Headline column: '{headline_column}' (used for semantic search)")
        print(f"Content column: '{content_column}' (stored as metadata)")
        print(f"Category column: '{category_column}' (stored as metadata)")
        print(f"Pooling method: CLS POOLING")
        print(f"Database: chroma_db_cls_headline")
        print(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data for ChromaDB
        ids = []
        embeddings = []
        metadatas = []
        documents = []

        for idx, row in df.iterrows():
            # Show progress every 500 articles (more frequent for 10k dataset)
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed
                eta = (total_articles - idx - 1) / articles_per_sec
                print(f"Processed {idx + 1}/{total_articles} articles "
                      f"({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)")

            # Get HEADLINE for semantic search
            headline_text = str(row[headline_column])

            # Skip empty headlines
            if len(headline_text.strip()) == 0:
                print(f"Warning: Skipping article {idx} - empty headline")
                continue

            try:
                # Generate embedding from HEADLINE using CLS pooling
                embedding = self.generate_embedding_for_text(headline_text)

                # Prepare data for ChromaDB
                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings.append(embedding.tolist())

                # Store HEADLINE as document (for display/search results)
                documents.append(headline_text)

                # Store metadata (content, category, and article index)
                content_text = str(row.get(content_column, ""))
                metadata = {
                    "article_index": idx,
                    "headline": headline_text,
                    "category": str(row.get(category_column, "Unknown")),
                    "content": content_text,  # Store full content in metadata
                    "content_length": len(content_text),
                    "pooling_method": "cls_pooling",  # Updated to CLS pooling
                    "search_column": "headline"  # Indicate what we're searching on
                }
                metadatas.append(metadata)

            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        # Add all embeddings to ChromaDB in batches (ChromaDB has max batch size limit)
        print(f"\n{'='*70}")
        print(f"STORING {len(ids)} EMBEDDINGS IN CHROMADB...")
        print(f"{'='*70}")

        # ChromaDB has a max batch size limit (~5000), so we add in batches
        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            self.collection.add(
                ids=ids[batch_idx:batch_end],
                embeddings=embeddings[batch_idx:batch_end],
                documents=documents[batch_idx:batch_end],  # Headlines stored as documents
                metadatas=metadatas[batch_idx:batch_end]
            )

        print(f"✓ All batches stored successfully!")

        total_time = time.time() - start_time
        print(f"\n✓ Successfully stored {len(ids)} embeddings in ChromaDB")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Average time per article: {total_time/len(ids):.4f} seconds")
        print(f"Processing speed: {len(ids)/total_time:.2f} articles/second")

    def search_similar_articles(self, query_text: str, n_results: int = 5) -> dict:
        """
        Search for similar articles using query text.

        Searches based on HEADLINE embeddings and returns results with
        headline, category, and full content.

        Args:
            query_text: Query text to find similar articles (matched against headlines)
            n_results: Number of similar articles to return

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding from headline using CLS pooling
        query_embedding = self.generate_embedding_for_text(query_text)

        # Search in ChromaDB
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about the ChromaDB collection.

        Returns:
            Dictionary with collection information
        """
        count = self.collection.count()
        return {
            "total_embeddings": count,
            "collection_name": self.collection.name,
            "pooling_method": "CLS POOLING",  # Updated to CLS pooling
            "embedding_dimension": 768,
            "search_column": "HEADLINE",
            "db_path": str(self.chroma_db_path)
        }

    def validate_cls_embeddings(self, sample_texts: list = None):
        """
        Validate that CLS pooling is working correctly by checking embeddings.

        Args:
            sample_texts: List of sample texts to test
        """
        if sample_texts is None:
            sample_texts = [
                "کرکٹ کی تازہ ترین خبریں",
                "پاکستانی سیاست کے حالیہ واقعات",
                "ٹیکنالوجی کی نئی دریافتیں"
            ]

        print(f"\n{'='*70}")
        print("VALIDATING CLS POOLING EMBEDDINGS")
        print(f"{'='*70}")

        for i, text in enumerate(sample_texts):
            embedding = self.generate_embedding_for_text(text)
            print(f"Sample {i+1}: '{text}'")
            print(f"  - Embedding shape: {embedding.shape}")
            print(f"  - Embedding norm: {np.linalg.norm(embedding):.4f}")
            print(f"  - Min value: {np.min(embedding):.4f}")
            print(f"  - Max value: {np.max(embedding):.4f}")
            print(f"  - Mean value: {np.mean(embedding):.4f}")
            print()


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # Load your preprocessed dataset
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM")
    print("USING CLS POOLING AND chroma_db_cls_headline DATABASE")
    print("SEMANTIC SEARCH ON HEADLINE COLUMN")
    print("="*70)
    print("\nLoading balanced dataset...")

    df = pd.read_csv("final_cleaned_urdu_news.csv")

    print(f"\nDataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset info:")
    print(f"  - Total articles: {len(df)}")
    print(f"  - Unique categories: {df['Category'].nunique()}")
    print(f"  - Categories: {df['Category'].unique().tolist()}")

    # Show sample data
    print(f"\n{'='*70}")
    print("SAMPLE DATA PREVIEW")
    print(f"{'='*70}")
    sample = df.iloc[0]
    print(f"Headline: {sample['Headline']}")  # Full headline since it's short
    print(f"Category: {sample['Category']}")
    print(f"Content preview: {sample['content'][:200]}...")
    print(f"Content length: {len(sample['content'])} characters")
    print(f"Headline length: {len(sample['Headline'])} characters")

    # Initialize embedding generator
    print(f"\n{'='*70}")
    print("INITIALIZING EMBEDDING GENERATOR")
    print(f"{'='*70}")

    embedder = UrduNewsEmbeddingGenerator(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_cls_headline"  # Updated path to cls_headline
    )

    # Validate CLS embeddings
    embedder.validate_cls_embeddings()

    # Generate embeddings and store in ChromaDB
    embedder.generate_embeddings_for_dataset(
        df=df,
        headline_column="Headline",    # Semantic search on HEADLINE (capital H)
        content_column="content",      # Store as metadata
        category_column="Category"     # Store as metadata (capital C)
    )

    # Display collection statistics
    print("\n" + "="*70)
    print("CHROMADB COLLECTION STATISTICS")
    print("="*70)
    stats = embedder.get_collection_stats()
    for key, value in stats.items():
        print(f"{key}: {value}")

    # Test: Search for similar articles with custom query
    print("\n" + "="*70)
    print("TESTING SEMANTIC SEARCH ON HEADLINE")
    print("="*70)

    # Custom query - search for headlines about mobile companies
    query = "موبائل کمپنیاں"
    print(f"\nQuery: {query}\n")

    results = embedder.search_similar_articles(
        query_text=query,
        n_results=5
    )

    print("Top 5 most relevant articles found:")
    print("="*70)

    if results['ids'] and len(results['ids']) > 0:
        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            print(f"\n{'='*70}")
            print(f"RESULT #{i+1}")
            print(f"{'='*70}")
            print(f"Article ID: {doc_id}")
            print(f"Similarity Score: {1 - distance:.4f} (higher is better)")
            print(f"\nHEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"Pooling Method: {metadata.get('pooling_method', 'cls_pooling')}")  # Updated
            print(f"Search Column: {metadata.get('search_column', 'headline')}")

            # Get full content from metadata (we stored it there)
            full_content = metadata.get('content', '')

            # Display article preview (first 300 characters)
            print(f"\n--- CONTENT PREVIEW (First 300 chars) ---")
            print(full_content[:300] + "..." if len(full_content) > 300 else full_content)

            print(f"\n{'='*70}")
    else:
        print("No results found!")

    # Test with multiple queries to compare performance
    test_queries = [
        "کرکٹ کی تازہ ترین خبریں",
        "پاکستانی سیاست کے حالیہ واقعات",
        "صحت اور ادویات کی نئی پالیسیاں"
    ]

    print("\n" + "="*70)
    print("ADDITIONAL QUERY TESTS")
    print("="*70)

    for test_query in test_queries:
        print(f"\nTesting query: '{test_query}'")
        test_results = embedder.search_similar_articles(test_query, n_results=3)

        if test_results['ids'] and len(test_results['ids']) > 0:
            for i, (doc_id, distance, document, metadata) in enumerate(zip(
                test_results['ids'][0],
                test_results['distances'][0] if 'distances' in test_results else [],
                test_results['documents'][0] if 'documents' in test_results else [],
                test_results['metadatas'][0] if 'metadatas' in test_results else []
            )):
                print(f"  {i+1}. {metadata.get('headline', 'N/A')} "
                      f"(Score: {1 - distance:.4f})")
        else:
            print("  No results found!")
        print()

    print("\n" + "="*70)
    print("✓ EMBEDDINGS GENERATION AND STORAGE COMPLETED SUCCESSFULLY!")
    print("✓ Using CLS POOLING for embeddings")
    print("✓ Database stored as: chroma_db_cls_headline")
    print("✓ Semantic search is now available on the HEADLINE column")
    print("✓ Content and categories are stored as metadata")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM
USING CLS POOLING AND chroma_db_cls_headline DATABASE
SEMANTIC SEARCH ON HEADLINE COLUMN

Loading balanced dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 4
  - Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters
Headline length: 59 characters

INITIALIZING EMBEDDING GENERATOR
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing ChromaDB at: chroma_db_cls_headline

VALIDATING CLS POOLING EMBEDDING

# **Recommender System Using CLS Pooling (Headline Embeddings)**

In [6]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM
# Uses pre-stored ChromaDB embeddings with CLS POOLING to generate recommendations
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path

class UrduNewsRecommender:
    """
    Recommendation system for Urdu news articles using pre-stored CLS POOLING embeddings.
    Connects to existing ChromaDB and generates recommendations based on queries.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "./chroma_db_cls_headline",
                 collection_name: str = "urdu_news_embeddings_10k_cls_headline"):
        """
        Initialize the recommender with pre-stored CLS POOLING embeddings.

        Args:
            model_name: HuggingFace model identifier (same as used for embedding generation)
            chroma_db_path: Path to existing ChromaDB with cls pooling embeddings
            collection_name: Name of the collection with cls pooling embeddings
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer (for query embeddings)
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Connect to existing ChromaDB with CLS POOLING embeddings
        print(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(path=str(self.chroma_db_path))

        # Get existing collection with CLS POOLING embeddings
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"✓ Connected to collection: {collection_name}")
            print(f"✓ Total articles in database: {self.collection.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name}'")
            print(f"Make sure embeddings are generated first!")
            raise e

    def cls_pooling(self, model_output, attention_mask):
        """
        Apply CLS POOLING to get sentence embeddings.

        This extracts the [CLS] token embedding which is specifically trained
        to represent the entire sequence.
        """
        token_embeddings = model_output[0]
        # Get the [CLS] token embedding (first token in the sequence)
        cls_embeddings = token_embeddings[:, 0, :]
        return cls_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for query text using CLS POOLING.

        Args:
            query_text: Urdu query text
            max_length: Maximum tokens per chunk
            chunk_overlap: Overlap between chunks

        Returns:
            Query embedding vector using CLS POOLING
        """
        # Tokenize to check length
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # If query fits in max_length, process normally
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.cls_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long queries: use chunking (same as article processing)
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.cls_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def get_recommendations(self, query: str, n_results: int = 5,
                          filter_category: str = None) -> dict:
        """
        Get article recommendations based on Urdu query using CLS POOLING embeddings.

        Args:
            query: Urdu text query
            n_results: Number of recommendations to return
            filter_category: Optional category filter (e.g., "Sports", "Business")

        Returns:
            Dictionary containing recommended articles with metadata
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS USING CLS POOLING")
        print(f"{'='*70}")
        print(f"Query: {query}")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Generate query embedding using CLS POOLING
        print("Generating query embedding using CLS POOLING...")
        query_embedding = self.generate_query_embedding(query)
        print("✓ Query embedding generated")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Search in ChromaDB with CLS POOLING embeddings
        print(f"Searching for similar articles...")
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"✓ Found {len(results['ids'][0]) if results['ids'] else 0} recommendations\n")

        return results

    def display_recommendations(self, results: dict, df: pd.DataFrame = None,
                               show_full_content: bool = False):
        """
        Display recommendations in a formatted way.

        Args:
            results: Results from get_recommendations()
            df: Optional dataframe to fetch full content
            show_full_content: Whether to display full article content
        """
        if not results['ids'] or len(results['ids'][0]) == 0:
            print("❌ No recommendations found!")
            return

        print(f"{'='*70}")
        print(f"TOP {len(results['ids'][0])} RECOMMENDATIONS")
        print(f"{'='*70}\n")

        for i, (doc_id, distance, document, metadata) in enumerate(zip(
            results['ids'][0],
            results['distances'][0] if 'distances' in results else [],
            results['documents'][0] if 'documents' in results else [],
            results['metadatas'][0] if 'metadatas' in results else []
        )):
            # Calculate similarity score (1 - distance for cosine)
            similarity_score = 1 - distance

            print(f"{'='*70}")
            print(f"RECOMMENDATION #{i+1}")
            print(f"{'='*70}")
            print(f"📰 Article ID: {doc_id}")
            print(f"🎯 Similarity Score: {similarity_score:.4f} ({similarity_score*100:.2f}%)")
            print(f"\n📌 HEADLINE: {metadata.get('headline', 'N/A')}")
            print(f"📂 CATEGORY: {metadata.get('category', 'N/A')}")
            print(f"📏 Content Length: {metadata.get('content_length', 'N/A')} characters")
            print(f"🔧 Pooling Method: {metadata.get('pooling_method', 'cls_pooling')}")

            # Get and display article content
            article_idx = metadata.get('article_index', None)

            if df is not None and article_idx is not None and article_idx < len(df):
                full_content = df.iloc[article_idx]['content']

                if show_full_content:
                    print(f"\n📄 FULL CONTENT:")
                    print(f"{'-'*70}")
                    print(full_content)
                else:
                    # Display preview (first 400 characters)
                    preview = full_content[:400] + "..." if len(full_content) > 400 else full_content
                    print(f"\n📄 CONTENT PREVIEW:")
                    print(f"{'-'*70}")
                    print(preview)
            else:
                # Fallback to stored document preview
                print(f"\n📄 CONTENT PREVIEW:")
                print(f"{'-'*70}")
                print(document)

            print(f"\n{'='*70}\n")

    def get_statistics(self) -> dict:
        """Get statistics about the recommendation system."""
        return {
            "total_articles": self.collection.count(),
            "collection_name": self.collection.name,
            "model": self.model_name,
            "device": str(self.device),
            "embedding_dimension": 768,
            "pooling_method": "CLS POOLING"
        }


# =============================================================================
# MAIN EXECUTION - RECOMMENDATION SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION SYSTEM - CLS POOLING EMBEDDINGS")
    print("="*70)

    # Load the dataset (optional - only needed for displaying full content)
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender (connects to existing ChromaDB with CLS POOLING embeddings)
    print("\n" + "="*70)
    print("INITIALIZING RECOMMENDATION SYSTEM WITH CLS POOLING")
    print("="*70)

    recommender = UrduNewsRecommender(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="./chroma_db_cls_headline",
        collection_name="urdu_news_embeddings_10k_cls_headline"
    )

    # Display system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    for key, value in stats.items():
        print(f"{key}: {value}")

    # =================================================================
    # EXAMPLE 1: Simple query-based recommendations
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 1: TECHNOLOGY NEWS RECOMMENDATION")
    print("="*70)

    query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"

    results = recommender.get_recommendations(
        query=query,
        n_results=5
    )

    recommender.display_recommendations(
        results=results,
        df=df,
        show_full_content=False  # Set to True to see full articles
    )

    # =================================================================
    # EXAMPLE 2: Category-filtered recommendations
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 2: SPORTS NEWS RECOMMENDATION (CATEGORY FILTERED)")
    print("="*70)

    query=" ن لائن ٹریفک نگرانی نئے حکومتی منصوبے خدشات "
    results = recommender.get_recommendations(
        query=query,
        n_results=10,
    )

    recommender.display_recommendations(
        results=results,
        df=df,
        show_full_content=False
    )

    # =================================================================
    # EXAMPLE 3: Multiple queries (batch recommendations)
    # =================================================================
    print("\n\n" + "="*70)
    print("EXAMPLE 3: MULTIPLE QUERY RECOMMENDATIONS")
    print("="*70)

    queries = [
        "معیشت اور کاروبار کی خبریں",
        "کرکٹ کی تازہ ترین خبریں",
        "فلموں اور ڈرامے کی خبریں",
        "پاکستان کا انتخابی نظام",
        "پاکستانی شوبز انڈسٹری",
        "صحت اور تندرستی کے حوالے سے مفید معلومات",
        "پاکستان میں تعلیمی نظام اور جدید تربیت",
        "ٹیکنالوجی",
        "کاروبار اور معاشی ترقی کی خبریں",
        "پاکستانی سیاست اور حکومتی پالیسیاں",
        "کھیلوں اور تفریحی پروگراموں کی خبریں",
        "مذہبی تعلیمات اور روحانی معلومات",
        "پاکستان کے خوبصورت سیاحتی مقامات"

    ]

    for idx, query in enumerate(queries, 1):
        print(f"\n{'─'*70}")
        print(f"QUERY {idx}: {query}")
        print(f"{'─'*70}")

        results = recommender.get_recommendations(
            query=query,
            n_results=10  # Get top 5 for each query
        )

        # Display only headlines for compact view
        if results['ids'] and len(results['ids'][0]) > 0:
            for i, (doc_id, distance, metadata) in enumerate(zip(
                results['ids'][0],
                results['distances'][0],
                results['metadatas'][0]
            ), 1):
                similarity = (1 - distance) * 100
                print(f"{i}. [{similarity:.1f}%] {metadata.get('headline', 'N/A')}")
        print()

    print("="*70)
    print("✓ RECOMMENDATION SYSTEM DEMO COMPLETED!")
    print("✓ Using CLS POOLING embeddings for recommendations")
    print("✓ Connected to chroma_db_cls database")
    print("="*70)

URDU NEWS RECOMMENDATION SYSTEM - CLS POOLING EMBEDDINGS

Loading dataset for content display...
✓ Dataset loaded: 111853 articles

INITIALIZING RECOMMENDATION SYSTEM WITH CLS POOLING
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Connecting to ChromaDB at: chroma_db_cls_headline
✓ Connected to collection: urdu_news_embeddings_10k_cls_headline
✓ Total articles in database: 111853

SYSTEM STATISTICS
total_articles: 111853
collection_name: urdu_news_embeddings_10k_cls_headline
model: urduhack/roberta-urdu-small
device: cuda
embedding_dimension: 768
pooling_method: CLS POOLING


EXAMPLE 1: TECHNOLOGY NEWS RECOMMENDATION

GENERATING RECOMMENDATIONS USING CLS POOLING
Query: پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن
Number of results: 5

Generating query embedding using CLS POOLING...
✓ Query embedding generated
Searching for similar articles...
✓ Found 5 recommendations

TOP 5 RECOMMENDATIONS

RECOMMENDATION #1
📰 Article ID: article_7
🎯 Similarity Score: 1